<a href="https://colab.research.google.com/github/abby1410/rag-hybrid-search-reranking/blob/main/hybrid_search_and_reranking_in_the_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.1/670.1 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 45.5 MB/s  0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.85.1
    Uninstalling grpcio-1.85.1:
      Successfully uninstalled grpcio-1.85.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [weaviate-client]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.85.1 requires grpcio>=1.85.1, but you have grpcio 1.78.2 which is incompatible.


In [2]:
!pip install langchain

In [5]:
!pip install -U langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.8 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
import weaviate

In [24]:
!pip install -q -U \
    weaviate-client \
    sentence-transformers \
    huggingface_hub \
    pypdf \
    pandas \
    numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [44]:
from google.colab import userdata
from huggingface_hub import login
import os

HF_TOKEN = userdata.get("HF_TOKEN")
WEAVIATE_URL = userdata.get("WEAVIATE_URL")
WEAVIATE_API_KEY = userdata.get("WEAVIATE_API_KEY")

assert HF_TOKEN, "Missing HF_TOKEN"
assert WEAVIATE_URL, "Missing WEAVIATE_URL"
assert WEAVIATE_API_KEY, "Missing WEAVIATE_API_KEY"

login(token=HF_TOKEN, add_to_git_credential=False)

print("Credentials configured successfully.")

Credentials configured successfully.


In [30]:
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length=512
)

print("Embedding and reranking models loaded.")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (13 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Embedding and reranking models loaded.


In [31]:
import weaviate
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

print("Weaviate ready:", client.is_ready())

Weaviate ready: True


In [32]:
from weaviate.classes.config import Configure, Property, DataType

COLLECTION_NAME = "RAGDocuments"

if client.collections.exists(COLLECTION_NAME):
    collection = client.collections.get(COLLECTION_NAME)
else:
    collection = client.collections.create(
        name=COLLECTION_NAME,

        vector_config=Configure.Vectors.self_provided(),

        properties=[
            Property(
                name="text",
                data_type=DataType.TEXT
            ),
            Property(
                name="source",
                data_type=DataType.TEXT
            ),
            Property(
                name="chunk_index",
                data_type=DataType.INT
            )
        ]
    )

print("Collection ready:", COLLECTION_NAME)

Collection ready: RAGDocuments


In [33]:
chunks = [
    {
        "text": "Hybrid search combines semantic and keyword retrieval.",
        "source": "rag_concepts.pdf",
        "chunk_index": 0
    },
    {
        "text": "BM25 ranks documents based on term frequency and rarity.",
        "source": "rag_concepts.pdf",
        "chunk_index": 1
    }
]

In [34]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 2
Embedding dimension: 384


In [35]:
with collection.batch.dynamic() as batch:

    for chunk, vector in zip(chunks, embeddings):

        batch.add_object(
            properties={
                "text": chunk["text"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            },
            vector=vector.tolist()
        )

if collection.batch.failed_objects:
    print("Failed objects:", len(collection.batch.failed_objects))
else:
    print("Documents uploaded successfully.")

Documents uploaded successfully.


In [36]:
from weaviate.classes.query import MetadataQuery

def hybrid_search(query, alpha=0.5, top_k=20):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    response = collection.query.hybrid(
        query=query,
        vector=query_embedding.tolist(),
        alpha=alpha,
        limit=top_k,
        return_metadata=MetadataQuery(score=True)
    )

    results = []

    for obj in response.objects:
        results.append({
            "text": obj.properties["text"],
            "source": obj.properties["source"],
            "chunk_index": obj.properties["chunk_index"],
            "hybrid_score": obj.metadata.score
        })

    return results

In [37]:
query = "How does hybrid search improve RAG retrieval?"

results = hybrid_search(
    query=query,
    alpha=0.5,
    top_k=10
)

for i, result in enumerate(results, 1):
    print(f"\nRank: {i}")
    print("Score:", result["hybrid_score"])
    print("Source:", result["source"])
    print("Text:", result["text"][:300])


Rank: 1
Score: 1.0
Source: rag_concepts.pdf
Text: Hybrid search combines semantic and keyword retrieval.

Rank: 2
Score: 0.0
Source: rag_concepts.pdf
Text: BM25 ranks documents based on term frequency and rarity.


In [38]:
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:

    results = hybrid_search(
        query="How does hybrid search improve RAG retrieval?",
        alpha=alpha,
        top_k=5
    )

    print(f"\nAlpha = {alpha}")

    for result in results:
        print(result["text"][:150])


Alpha = 0.0
Hybrid search combines semantic and keyword retrieval.
BM25 ranks documents based on term frequency and rarity.

Alpha = 0.25
Hybrid search combines semantic and keyword retrieval.
BM25 ranks documents based on term frequency and rarity.

Alpha = 0.5
Hybrid search combines semantic and keyword retrieval.
BM25 ranks documents based on term frequency and rarity.

Alpha = 0.75
Hybrid search combines semantic and keyword retrieval.
BM25 ranks documents based on term frequency and rarity.

Alpha = 1.0
Hybrid search combines semantic and keyword retrieval.
BM25 ranks documents based on term frequency and rarity.


In [39]:
def rerank_results(query, retrieved_results, top_k=5):

    if not retrieved_results:
        return []

    pairs = [
        (query, result["text"])
        for result in retrieved_results
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(retrieved_results, scores):
        item = result.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [40]:
query = "How does hybrid search improve RAG retrieval?"

retrieved = hybrid_search(
    query=query,
    alpha=0.5,
    top_k=20
)

reranked = rerank_results(
    query=query,
    retrieved_results=retrieved,
    top_k=5
)

for rank, result in enumerate(reranked, 1):
    print(f"\nRank: {rank}")
    print("Rerank score:", result["rerank_score"])
    print("Source:", result["source"])
    print("Text:", result["text"][:300])


Rank: 1
Rerank score: 1.457507848739624
Source: rag_concepts.pdf
Text: Hybrid search combines semantic and keyword retrieval.

Rank: 2
Rerank score: -11.401573181152344
Source: rag_concepts.pdf
Text: BM25 ranks documents based on term frequency and rarity.


In [41]:
evaluation_set = [
    {
        "query": "What is BM25?",
        "relevant_chunks": [1, 4]
    },
    {
        "query": "How does semantic search work?",
        "relevant_chunks": [2, 5]
    }
]

In [42]:
def reciprocal_rank(results, relevant_chunks):

    for rank, result in enumerate(results, 1):

        if result["chunk_index"] in relevant_chunks:
            return 1 / rank

    return 0


def evaluate_retrieval(search_function, evaluation_set):

    scores = []

    for item in evaluation_set:

        results = search_function(item["query"])

        score = reciprocal_rank(
            results,
            item["relevant_chunks"]
        )

        scores.append(score)

    return sum(scores) / len(scores)